# Azure Databricks의 컴퓨팅 선택 및 구성

## 🏥 의료 산업 시나리오
**MedTech Analytics Corporation**은 매일 수백만 건의 환자 기록, 검사 결과, 의료 기기 원격 측정 데이터를 처리하는 의료 데이터 분석 회사입니다. Azure Databricks 플랫폼은 다음과 같은 다양한 워크로드를 지원합니다:
- **실시간 환자 모니터링**: IoT 의료 기기에서 스트리밍되는 데이터
- **임상 데이터 웨어하우징**: EHR(Electronic Health Records) 처리 ETL 파이프라인
- **의료 연구 분석**: 데이터 과학자의 임시 분석
- **규정 보고**: HIPAA 준수 보고서 자동 생성
- **예측 의료 모델**: 조기 진단을 위한 머신 러닝이 Notebook에서는 보안, 비용 효율성, 성능을 유지하면서 다양한 의료 데이터 엔지니어링 워크로드에 최적화된 컴퓨팅 리소스를 선택하고 구성하는 방법을 보여줍니다.
- **설정**: 설정 스크립트를 실행하여 샘플 의료 데이터 테이블을 만듭니다.

In [0]:
from scripts.setup import setup, setup_02

spark.sql("USE CATALOG trainer_demo")
setup_02(spark)

## 적절한 컴퓨팅 유형 선택

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/select-and-configure-compute.choose-appropriate-compute-type.png)

다양한 컴퓨팅 유형은 서로 다른 시나리오에 적합합니다. 다음 표는 정보에 입각한 결정을 내리는 데 도움이 되도록 주요 특성을 비교합니다:

| 컴퓨팅 유형                | 권장 용도                                 | 시작 시간   | 관리 오버헤드                  | 비용 효율성                          | 주요 제한 사항                        |
|---------------------------|-------------------------------------------|-------------|-------------------------------|---------------------------------------|----------------------------------------|
| Serverless compute        | 대화형 개발, ETL 작업, BI 워크로드        | ⚡ 2-6초     | 👌 최소 - 완전 관리형           | 🟢 높음 - 0까지 확장, 사용한 만큼만 지불 | RDD API, R 또는 JAR 라이브러리 미지원  |
| Classic compute (Standard)| 협업형 데이터 엔지니어링, 공유 분석       | ⏱️ 3-7분    | 🔧 보통 - 구성 및 모니터링 필요  | 🟡 보통 - 다중 사용자 공유로 사용자당 비용 감소 | 거버넌스를 위해 Unity Catalog 필요     |
| Classic compute (Dedicated)| RDD 워크로드, GPU 작업, R 언어, 사용자 지정 컨테이너 | ⏱️ 3-7분    | 🔧 보통 - 구성 및 모니터링 필요  | 🔴 낮음 - 단일 사용자/그룹 전용         | 공유 리소스보다 비용이 높음            |
| Instance pools            | 빠른 시작이 필요한 자주 실행되는 클래식 워크로드 | 🚀 1분 미만  | ⚠️ 높음 - 유휴 용량 유지 필요    | 🔄 변동적 - 사용 빈도로 유휴 비용 정당화 | 유휴 인스턴스 비용 발생               |
| SQL warehouse (Serverless)| SQL 분석, BI 대시보드, 보고               | ⚡ 2-6초     | 👌 최소 - 지능형 워크로드 관리   | 🟢 높음 - Photon + Predictive IO와 함께 동적 확장 | SQL 워크로드만 지원                   |
| SQL warehouse (Pro)       | 사용자 지정 네트워킹, 페더레이션이 필요한 SQL | ⏱️ 약 4분   | 🔧 보통 - 수동 크기 조정 구성    | 🟡 보통 - Photon + Predictive IO      | Serverless보다 느린 확장              |
| SQL warehouse (Classic)   | 입문 수준 SQL 탐색                         | ⏱️ 약 4분   | 🔧 보통 - 수동 크기 조정 구성    | 🔴 낮음 - 기본 Photon만 제공           | 제한된 성능 기능                      |
| Job compute (Serverless)  | 자동화된 워크플로, 프로덕션 ETL           | ⚡ 2-6초     | 👌 최소 - 완료 후 자동 종료      | 🟢 높음 - 유휴 비용 없음               | Serverless compute와 동일             |
| Job compute (Classic)     | 사용자 지정 구성이 필요한 작업             | ⏱️ 3-7분    | 🔧 보통 - 정책 구성 필요         | 🟡 보통 - 자동 종료로 유휴 낭비 방지   | 인프라 관리 필요                      |

## 컴퓨팅 성능 구성

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/select-and-configure-compute.configure-compute-performance.png)

![diagram](https://raw.githubusercontent.com/asddai/AzureDatabricks202606/main/Demo/infographics/KO/KO-instance-pool-management.png)

### 🎯 데모: 의료 워크로드에 대한 노드 유형 선택다양한 의료 워크로드는 리소스 요구 사항에 따라 서로 다른 VM 인스턴스 유형의 이점을 얻습니다.

**의료 워크로드 → 노드 유형 매핑:**

| 워크로드                        | 노드 유형                  | 이유                                                         |
|----------------------------------|----------------------------|--------------------------------------------------------------|
| **검사 결과 집계**<br>(대규모 조인)         | **메모리 최적화(E 시리즈)**     | 수백만 건의 검사 결과를 환자 기록과 조인                      |
| **의료 기기 원격 측정 ETL**<br>(단순 변환)   | **컴퓨팅 최적화(F 시리즈)**     | 단순 변환, 높은 CPU 사용량                                   |
| **유전체 데이터 분석**<br>(빠른 로컬 I/O)    | **스토리지 최적화(L 시리즈)**    | 대용량 유전체 시퀀스 파일을 반복적으로 스캔                   |
| **딥러닝 - 의료 영상**<br>(GPU 필요)         | **GPU 인스턴스(NC/ND 시리즈)**   | X-ray/MRI 이미지로 CNN 학습                                  |

**데모 설정:**
1. **Compute** → **Create Compute**로 이동
2. 워크로드에 맞는 인스턴스 유형 선택
3. 워커 수 및 자동 크기 조정 구성

### 🎯 데모: 자동 크기 조정 구성

**의료 사용 사례: 가변 부하 ETL 파이프라인**

MedTech의 시간별 파이프라인은 의료 기기 데이터의 처리량이 시간대에 따라 달라집니다:

- **비혼잡 시간대 (오전 2-6시)** 시간당 1만 레코드 → 워커 2개 필요
- **피크 시간대 (오전 8시 - 오후 5시)** 시간당 50만 레코드 → 워커 8개 필요
- **주말**  데이터 최소 → 워커 1개 필요

**자동 크기 조정 구성:**
```
Min workers: 2
Max workers: 8
Autoscaling mode: Optimized
```


**이점:**
- 의료 기기 데이터가 급증할 때 확장
- 비혼잡 시간대에는 축소
- 고정 8-워커 클러스터 대비 40-60% 비용 절감

### 🎯 데모: 자동 종료 설정

**자동 종료로 컴퓨팅 비용 낭비를 방지합니다:**

| 사용 사례                | 종료 설정      | 이유                                 |
|-------------------------|---------------|--------------------------------------|
| 데이터 과학자 탐색       | 45분          | 쿼리 사이 분석 검토 시간 확보         |
| 공유 팀 클러스터         | 60분          | 협업 작업을 위한 더 긴 시간 제한      |
| 작업 클러스터           | 완료 후        | 자동(수동 설정 불필요)                |
| 프로덕션 장기 실행       | 사용 안 함     | 중요한 워크로드 24/7 가용성           |

**구성 방법:**
1. **Compute** → **Edit Cluster**로 이동
2. **Advanced Options** → **Automatic Termination** 선택
3. 분 단위로 종료 시간 설정 (예: 45)

### 🎯 데모: 비용 최적화를 위한 스팟 인스턴스

**MedTech의 비용 절감 전략:**

| 노드 유형      | 인스턴스 유형           | 이유                                   |
|---------------|------------------------|----------------------------------------|
| 워커 노드     | Spot 인스턴스 (Azure Spot VM) | 최대 90% 비용 절감<br>Spark가 중단 자동 처리<br>종료 전 디커미셔닝으로 데이터 이동 |
| 드라이버 노드 | 온디맨드 인스턴스       | 클러스터 안정성 보장<br>드라이버 실패 시 전체 클러스터 실패 |

**구성 예시:**

```
Worker instances: Azure Spot VMs
Driver instance: On-demand
Enable decommissioning: Yes
```


**적합한 경우:**
- 배치 ETL 작업
- 시간 민감도가 낮은 분석
- 개발/테스트 워크로드

**피해야 할 경우:**
- 실시간 환자 모니터링
- SLA가 중요한 애플리케이션
- 대화형 임원 대시보드

### 🎯 데모: 인스턴스 풀 사용

**MedTech가 인스턴스 풀을 사용하는 경우:**
데이터 엔지니어링 팀은 개발 중에 클러스터를 자주 만들고 삭제합니다:
- 하루 20회 클러스터 생성
- 표준 시작 시간: 5분
- 풀 사용 시: 1분 미만

**풀 구성:**
```
Min idle instances: 3
Max capacity: 10
Idle termination: 30 minutes
Preloaded runtime: 14.3 LTS
```

**비용 대비 효과:**
- 팀이 하루에 5개 이상 클러스터를 만들 때 정당화 가능
- 유휴 비용: VM 3대 × 24시간 = 하루 72 VM-hour
- 절감: 20개 클러스터 × 4분 절감 = 80분의 생산성 향상

**풀을 만들려면:**
1. **Compute** → **Instance Pools** → **Create**
2. 최소/최대 인스턴스 구성
3. VM 유형 선택
4. 런타임 버전 미리 로드
5. 풀에서 클러스터 생성

## 컴퓨팅 기능 구성

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/select-and-configure-compute.configure-compute-feature-settings.png)

### 🎯 데모: Photon 가속

**Photon**은 네이티브 코드 실행을 사용하여 SQL 및 DataFrame 작업을 가속합니다.

**MedTech가 Photon을 사용하는 경우:**
- ✅ 100개 이상의 열이 있는 넓은 테이블 스캔
- ✅ 환자 테이블과 청구 테이블 간 대규모 조인
- ✅ 수백만 건의 검사 결과에 대한 복잡한 집계
- ❌ 1000행 미만의 작은 데이터셋에 대한 단순 변환
- ❌ GPU 기반 딥러닝 워크로드

**Photon을 사용하려면:**
1. 클러스터 생성 중 **Performance** 섹션을 펼칩니다
2. 런타임 버전 9.1 LTS 이상을 선택합니다
3. **Photon Acceleration**을 **Enabled**로 전환합니다
4. **참고**: Photon은 GPU 클러스터와 호환되지 않습니다

### 🎯 데모: Databricks Runtime 선택

**MedTech의 런타임 전략:**

| 클러스터 유형         | 런타임                  | 이유                                         |
|----------------------|------------------------|----------------------------------------------|
| **개발 클러스터**     | 최신 표준 런타임         | 최신 기능 및 최적화 사용                     |
| **ML 학습 클러스터**  | Databricks Runtime ML   | 사전 설치된 ML 라이브러리(TensorFlow, PyTorch)|
| **프로덕션 ETL 작업** | LTS(Long-Term Support)  | 안정성, 확장된 호환성, 검증된 코드           |
| **레거시 파이프라인** | 이전 LTS                | 마이그레이션 완료 전까지 호환성 유지         |

**예시 런타임:**
- Standard: `14.3 LTS` (Apache Spark 3.5.0)
- ML: `14.3 LTS ML` (scikit-learn, TensorFlow, PyTorch, MLflow 포함)

## 컴퓨팅용 라이브러리 설치

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/select-and-configure-compute.install-libraries-for-compute.png)

![library installation diagram](https://raw.githubusercontent.com/asddai/AzureDatabricks202606/main/Demo/infographics/KO/KO-library-installation.png)

### 🎯 데모: 의료 워크로드용 라이브러리 설치

**MedTech의 라이브러리 요구 사항:**

| 사용 사례             | 라이브러리         | 설치 방법           |
|----------------------|-------------------|---------------------|
| **FHIR 데이터 파싱**   | `fhir.resources`  | PyPI                |
| **SQL Server 연결**   | `pymssql`         | PyPI(특정 버전)     |
| **의료 코딩**         | `icd10-cm`        | PyPI                |
| **SQL용 JDBC 드라이버**| `mssql-jdbc`      | Maven               |
| **사용자 지정 HL7 파서**| 내부 wheel 파일   | Unity Catalog 볼륨  |

**설치 수준:**
- **Compute 범위**: 클러스터의 모든 Notebook에서 사용 가능(영구적)
- **Notebook 범위**: 특정 Notebook에서만 사용 가능(임시)

### 🎯 데모: PyPI에서 설치

**시나리오**: MedTech가 환자 인구 통계 데이터를 쿼리하기 위해 SQL Server에 연결해야 합니다.

**라이브러리**: `pymssql` (Python SQL Server 드라이버)

**단계:**
1. **Compute** → 클러스터 선택
2. **Libraries** 탭 클릭
3. **Install new** 클릭
4. **PyPI** 선택
5. 패키지 이름 입력: `pymssql==2.3.1` (안정성을 위한 고정 버전)
6. **Install** 클릭

**권장 사례**: 프로덕션 워크로드에서는 재현성을 보장하기 위해 항상 버전 번호를 지정합니다.

### 🎯 데모: Maven에서 설치

**시나리오**: 더 나은 성능을 위해 JDBC를 사용하여 Azure SQL에 연결합니다.

**라이브러리**: Microsoft JDBC Driver for SQL Server

**Maven 좌표:**  
`com.microsoft.sqlserver:mssql-jdbc:12.8.1.jre11`

**단계:**  
1. **Compute** → **Libraries** → **Install new**  
2. **Maven** 선택  
3. 좌표 입력: `com.microsoft.sqlserver:mssql-jdbc:12.8.1.jre11`  
4. **Install** 클릭

**참고**: Standard access mode 클러스터에서는 allowlist에 Maven 좌표를 추가해야 합니다.

### 🎯 데모: Unity Catalog 볼륨에서 설치

**시나리오**: MedTech는 HL7 메시지 파싱을 위한 사용자 지정 Python 라이브러리를 개발했습니다.

**설정:**
1. 공유 라이브러리를 위한 볼륨 생성
2. 볼륨에 wheel 파일 업로드
3. 볼륨 경로에서 설치

**장점:**
- Unity Catalog 권한으로 설치 가능 사용자 제어
- 감사 로그로 라이브러리 사용 추적
- 워크스페이스 전체에서 중앙 집중식 관리

In [0]:
%sql
-- Create a volume for storing custom libraries

CREATE VOLUME IF NOT EXISTS trainer_demo.demo_02.shared_libraries
  COMMENT 'Shared Python libraries for healthcare analytics';

-- Grant read access to data engineering team
-- GRANT READ VOLUME ON VOLUME trainer_demo.demo_02.shared_libraries TO `data_engineers`;

SELECT 'Volume created: /Volumes/trainer_demo/demo_02/shared_libraries' as status;

**라이브러리를 볼륨에 업로드:**
1. **Catalog Explorer**로 이동
2. `trainer_demo.demo_02.shared_libraries`로 이동
3. **Upload to this volume** 클릭
4. `hl7_parser-1.2.3-py3-none-any.whl` 업로드 (빈 파일도 가능)

**볼륨에서 설치:**
1. **Compute** → **Libraries** → **Install new**
2. **Python Whl** 선택
3. 경로 입력: `/Volumes/trainer_demo/demo_02/shared_libraries/hl7_parser-1.2.3-py3-none-any.whl`
4. **Install** 클릭

**보안:** Standard access mode에서는 볼륨 경로를 allowlist에 추가해야 합니다.

### 🎯 데모: requirements.txt 사용하기

**시나리오**: 여러 의료 분석 라이브러리를 한 번에 설치합니다.

**requirements.txt 만들기:**
txt
fhir.resources==7.1.0
icd10-cm==0.0.4
numpy==1.24.3
pandas==2.1.0
sqlalchemy==2.0.21


**업로드 대상:**
- Workspace Files: `/Workspace/Users/you@example.com/requirements.txt`
- Unity Catalog Volume: `/Volumes/trainer_demo/demo_02/shared_libraries/requirements.txt`

**설치:**
1. **Compute** → **Libraries** → **Install new**
2. **Requirements** 선택
3. requirements.txt 경로 입력
4. **Install** 클릭 → 모든 패키지가 자동으로 설치됨

### 🎯 데모: init 스크립트(고급)

**시스템 수준 구성에는 init 스크립트를 사용하고, 라이브러리 설치에는 사용하지 마세요.**

**유효한 사용 사례:**
- 시스템 패키지 설치 (`apt-get install`)
- 환경 변수 구성
- 모니터링 에이전트 설정
- 시스템 라이브러리가 필요한 데이터베이스 드라이버 설치

**예제: 의료 데이터 암호화 설정 구성**

```bash
#!/bin/bash
# File: /Volumes/trainer_demo/demo_02/scripts/security_init.sh

# HIPAA 준수 환경 변수 설정
export ENABLE_ENCRYPTION=true
export AUDIT_LOGGING=enabled

# 데이터베이스 연결용 TLS 구성
echo "TLS_REQCERT=strict" >> /etc/openldap/ldap.conf

echo "Security configuration complete"
```

**클러스터를 init 스크립트로 실행하도록 구성:**
1. **Compute** → **Edit** → **Advanced options** → **Init Scripts**
2. 경로 추가: `/Volumes/trainer_demo/demo_02/scripts/security_init.sh`
3. Standard access mode: 먼저 경로를 allowlist에 추가

### 🎯 데모: Standard 액세스 모드 허용 목록

**Standard access mode를 사용하는 공유 클러스터의 경우:**

**Metastore 관리자는 다음을 allowlist에 추가해야 합니다:**
- Maven 좌표
- JAR 파일 경로
- init 스크립트 경로

**허용 목록 예시 항목:**

```
Maven: com.microsoft.sqlserver:mssql-jdbc
Volume path: /Volumes/trainer_demo/demo_02/shared_libraries/
Init scripts: /Volumes/trainer_demo/demo_02/scripts/
```


**허용 목록 구성(관리자만 가능):**
1. **Catalog Explorer** → **Metastore** 선택
2. **Settings** → **Allowed JARs/Init Scripts**
3. **Add** 클릭
4. Maven 좌표 또는 경로 입력
5. 저장

**장점:**
- 보안 팀이 승인된 라이브러리 통제
- 승인되지 않은 코드 실행 방지
- 감사 추적 유지
- 가드레일 내에서 셀프 서비스 가능

## 컴퓨팅 액세스 구성

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/select-and-configure-compute.configure-compute-access-perms.png)

### 🎯 데모: 컴퓨팅 권한 수준

**MedTech의 권한 전략:**

| 역할                  | 권한 수준         | 가능한 작업                       | 불가능한 작업           |
|----------------------|------------------|-----------------------------------|-------------------------|
| **팀 리드**           | `CAN MANAGE`     | 전체 제어: 편집, 라이브러리, 권한 | —                      |
| **시니어 데이터 엔지니어** | `CAN RESTART`    | 클러스터 연결 + 재시작            | 구성 편집, 권한 관리    |
| **데이터 분석가**      | `CAN ATTACH TO`  | 쿼리 실행, Spark UI 보기           | 재시작, 구성 편집       |
| **권한 없음**         | 없음             | —                                 | 클러스터를 볼 수 없음   |

**권한 계층 구조:**
```
NO PERMISSIONS
    ↓
CAN ATTACH TO (메트릭 보기, Notebook 실행)
    ↓
CAN RESTART (+ 클러스터 수명 주기 제어)
    ↓
CAN MANAGE (+ 구성 편집, 라이브러리 설치, 권한 관리)
```

### 🎯 데모: 컴퓨팅 권한 구성

**시나리오**: 공유 분석 클러스터에 **Healthcare Analytics 팀**의 액세스 권한을 부여합니다.

**단계:**
1. **Compute** → 클러스터 선택
2. **Permissions** 클릭(세 점 메뉴 또는 Permissions 탭)
3. **Add** 클릭
4. 그룹 검색: `healthcare_analysts`
5. 권한 선택: `CAN ATTACH TO`
6. **Save** 클릭

**결과**: 이제 `healthcare_analysts` 그룹의 모든 구성원은 다음이 가능합니다:
- 노트북을 클러스터에 연결
- 클러스터 메트릭 및 Spark UI 보기
- 쿼리 및 분석 실행
- 하지만 클러스터를 재시작하거나 수정할 수는 없음

**권장 사례: 개별 사용자 대신 그룹 사용**

❌ **하지 마세요:**
```
alice@medtech.com → CAN ATTACH TO
bob@medtech.com → CAN ATTACH TO
charlie@medtech.com → CAN ATTACH TO
```

✅ **이렇게 하세요:**
```
healthcare_analysts (group) → CAN ATTACH TO
  ├─ alice@medtech.com
  ├─ bob@medtech.com
  └─ charlie@medtech.com
```

**장점:**
- 팀 변화에 따라 관리가 쉬움
- 리소스 전반에 걸친 일관된 권한
- 새 팀원에 대한 자동 액세스
- 더 간단한 감사 및 규정 준수

### 🎯 데모: 워크스페이스 엔타이틀먼트

**제한 없는 클러스터 생성 엔타이틀먼트:**  
기본적으로 관리자인 경우에만 클러스터를 만들 수 있습니다.  
이 엔타이틀먼트를 부여하면 비관리자도 크기 제한 없이 클러스터를 만들 수 있습니다.

**MedTech 예시:**  
- 데이터 엔지니어: ✅ 제한 없는 클러스터 생성  
- 데이터 분석가: ❌ 사전 구성된 공유 클러스터만 사용  
- 데이터 과학자: ✅ 제한 없음(ML 실험용)

**부여 방법:**  
1. **Settings** → **Identity and access** → **Users** 또는 **Groups**  
2. 사용자/그룹 선택  
3. **Entitlements** → **Allow cluster creation** 활성화  
4. 저장